In [69]:
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

Cars_df = pd.read_csv("/content/drive/MyDrive/cars_clean.csv")

In [88]:
# The number of null rows in the 'color' column is less than 5% of the total row count, so we are going to drop them.
Cars_df = Cars_df[~Cars_df['color'].isna()]

In [127]:
#For the 'km' column, the number of missing values is too large to simply drop the rows.
# We are filling the missing values with the median to prevent outliers from skewing the data distribution.
Cars_df['km'] = Cars_df['km'].fillna(Cars_df['km'].median())

/tmp/ipykernel_4674/34523093.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [128]:
Cars_df.isna().sum()

,0
id,0
make,0
model,0
year,0
km,0
price_egp,0
transmission,0
color,0
city,0
seller_type,0


In [126]:
px.bar(Cars_df.groupby(['fuel_type'])['price_egp'].median().reset_index(),x='fuel_type',y='price_egp')

### Price Distribution by Fuel Type

* **Premium Fuel Segments**:
  * **Hybrid**: Has the highest median price at approximately **2.7M EGP**.
  * **Electric**: Follows as the second most expensive at approximately **1.5M EGP**.
* **Budget/Standard Fuel Segments**:
  * **Petrol, Diesel, & CNG**: Much more affordable and closely priced, ranging from **0.5M to 0.7M EGP**.
  * **CNG**: Represents the most economical option overall.

* **Key Takeaway**: A typical hybrid vehicle is listed at roughly **four times** the price of a typical petrol vehicle.

In [124]:
px.bar(Cars_df.groupby(['seller_type'])['price_egp'].median().reset_index(),x='seller_type',y='price_egp')

### Price Distribution by Seller Type

* **Dealers**: List vehicles at a median price of approximately **1.2M EGP**.
* **Private Sellers**: List vehicles at a median price of approximately **0.65M EGP**.

* **Key Takeaway**: Dealer listings carry a median price tag that is nearly **double** (approximately 1.85x) that of private listings.

In [125]:
px.bar(Cars_df.groupby(['transmission'])['price_egp'].median().reset_index(),x='transmission',y='price_egp')

### Key Insight

* **Transmission Price Comparison**: The average price for a **manual** transmission car is significantly lower than that of an **automatic** one.

In [98]:
px.histogram(Cars_df.groupby(['model'])['price_egp'].mean().reset_index().sort_values('price_egp',ascending=False),x='model',y='price_egp',histfunc='avg')

### Price Distribution by Car Model

* **Highest Average Prices**: Models like the **Land Cruiser**, **Patrol**, **Armada**, and **Prado** show the highest average prices.
* **Lowest Average Prices**: Models like the **Starlet**, **Excel**, **Pulsar**, and **Datsun** show the lowest average prices.

In [129]:
g = ["make", "model", "year"]
df = Cars_df[Cars_df.groupby(g).price_egp.transform("size") >= 5].copy()
df["rel_price"] = df.price_egp / df.groupby(g).price_egp.transform("median")

c = df.groupby("color").agg(rel_price=("rel_price", "median"), n=("rel_price", "count")).reset_index()
c = c[c.n >= 30]   # rare colors are too noisy

px.bar(c.sort_values("rel_price", ascending=False), x="color", y="rel_price", text="n"
      ).add_hline(y=1, line_dash="dash").update_yaxes(range=[0.9, 1.1]).show()


### Impact of Car Color on Relative Price

* **Stable Pricing Across Common Colors**: Standard colors (such as **Black**, **Silver**, **White**, **Gray**, and **Blue**) have median relative prices tightly centered at exactly **1.0** (meaning they align perfectly with the overall model-year median).
* **Slight Deviations for Less Common Colors**:
  * **Dark Red**: Shows a slight price premium of **~2.5%** above the median.
  * **Mocha & Petroleum**: Show minor discounts of **~1.6%** to **~1.7%** below the median.

* **Key Takeaway**: Vehicle color has a very minor impact on relative market pricing, as almost all colors sit within **±2.5%** of the standard market median.

In [130]:
s = df.groupby("city").agg(rel_price=("rel_price", "median"), availability=("rel_price", "count")).reset_index()
s = s[s.availability >= 5]

px.scatter(s, x="availability", y="rel_price", hover_name="city", log_x=True, opacity=0.6,
           trendline="ols", trendline_options=dict(log_x=True)
          ).add_hline(y=1, line_dash="dash").show()

### Market Availability vs. Relative Price Index

* **No Correlation Observed**: A city's listing availability shows no significant relationship with how its vehicle prices compare to the regional medians of identical models.
* **Conclusion**: High-volume markets (greater listing availability) do not exhibit systematically cheaper or more expensive relative pricing compared to smaller local markets.

In [97]:
# Getting Outliers by model and year and make
px.box(df, y="rel_price", log_y=True, points="outliers",
       hover_data=["make", "model", "year", "price_egp", "city","km"]).show()
